In [2]:
from torch.utils.data import Dataset, DataLoader
import numpy as np 
import torch 
import os 
data_path = os.path.join('data','UBFC-RPPG-Dataset')
subjects = os.listdir(data_path)  

In [3]:
sample_subject = subjects[0]
sample_roi = np.loadtxt(os.path.join(data_path, sample_subject, 'roi_colors.txt'), delimiter=',')
sample_roi.shape  

(2035, 9)

In [4]:
sample_signal = np.loadtxt(os.path.join(data_path, sample_subject, 'ground_truth.txt'))

In [5]:
#each frame has 3 labels in the dataset : Normalized Blood Volume Pulse , Heart Rate and Time. We will only train on the first one


class UBFC_Dataset(Dataset):
    def __init__(self, data_path, subjects,seq_len=10):
        self.data_path = data_path 
        self.subjects = subjects 
        self.seq_len = seq_len
        self.possible_ranges = []
        for subject in subjects:
            signal_path = os.path.join(data_path,subject, 'ground_truth.txt')
            signal = np.loadtxt(signal_path)
            num_starts = signal.shape[-1] - seq_len 
            for i in range(num_starts): 
                self.possible_ranges.append((subject, i)) 
    def __len__(self):
        return len(self.possible_ranges)

    def __getitem__(self, index):
        subject,i = self.possible_ranges[index]
        signal_path = os.path.join(self.data_path,subject, 'ground_truth.txt')
        colors_path = os.path.join(self.data_path, subject, 'roi_colors.txt')
        signals = np.loadtxt(signal_path)
        colors = np.loadtxt(colors_path, delimiter=',')
        signal_seq = signals[0,i : i + self.seq_len]
        color_seq = colors[i : i + self.seq_len]


        # --- MADE A BIG MISTAKE OF JUST NORMALIZING BY DIVIDING BY 255 AND THAT WAS CAUSING SATURATION
        
        min_vals = color_seq.min(axis=0, keepdims=True)
        max_vals = color_seq.max(axis=0, keepdims=True)
        color_seq = (color_seq - min_vals) / (max_vals - min_vals + 1e-6)
        # -------------------------------
        return torch.tensor(color_seq, dtype=torch.float32), torch.tensor(signal_seq, dtype=torch.float32)


In [6]:
batch_size = 32
seq_len = 128
dataset = UBFC_Dataset(data_path, subjects,seq_len=seq_len)

from  torch.utils.data import random_split


  


train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size 
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle=False)

sample_color_seq, sample_signal_seq = dataset[0]
print(sample_color_seq.shape)

torch.Size([128, 9])


In [7]:
class LSTMModel(torch.nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_size, hidden_size, num_layers,batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, output_size) 
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        output = self.fc(lstm_out)  
        return output  

In [8]:
model = LSTMModel(input_size=9, hidden_size=64, num_layers=2, output_size=1)
criterion = torch.nn.MSELoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
sum([param.numel() for param in model.parameters()])

52545

In [ ]:
log_every = 10
num_epochs = 10 
for epoch in range(num_epochs):
    model.train()
    total_loss = 0  
    for i, (color_seq, signal_seq) in enumerate(train_loader):
        optimizer.zero_grad()
        predictions = model(color_seq) 
        # print(predictions.shape)
        # print(signal_seq.shape)
        # break 
    # break
        loss = criterion(predictions.flatten(), signal_seq.flatten())
        loss.backward()
        if (i+1) % log_every == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
        optimizer.step() 
        